# Ranked list preparation

- Take gs://genetics-portal-dev-analysis/dc16/output/genes_pleiotropy
- Make rank list of genes based on TA pleiotropy count
- Apply z-score correction

In [4]:
from pathlib import Path

from gentropy.common.session import Session

gcs_genes_pleiotropy = "gs://genetics-portal-dev-analysis/dc16/output/genes_pleiotropy"
filepath = str(Path("../../../data/genes_pleiotropy").resolve())

session = Session(extended_spark_conf={"spark.driver.memory": "10g"})

In [ ]:
# Download genes_pleiotropy parquet from GCS to data folder (run once; requires gcloud auth)
import subprocess

local_dir = Path("../../../data/genes_pleiotropy").resolve()
local_dir.mkdir(parents=True, exist_ok=True)
subprocess.run(
    [
        "gcloud",
        "storage",
        "--billing-project=open-targets-genetics-dev",
        "rsync",
        "-r",
        gcs_genes_pleiotropy,
        str(local_dir),
    ],
    check=True,
)

At gs://genetics-portal-dev-analysis/dc16/output/genes_pleiotropy/**, worker process 42776 thread 8625679424 listed 12...
Copying gs://genetics-portal-dev-analysis/dc16/output/genes_pleiotropy/._SUCCESS.crc to file:///Users/polina/Gentropy-manuscript/data/genes_pleiotropy/._SUCCESS.crc
Copying gs://genetics-portal-dev-analysis/dc16/output/genes_pleiotropy/.part-00000-bc3ff23c-243e-4360-a61a-61cb7c42af54-c000.snappy.parquet.crc to file:///Users/polina/Gentropy-manuscript/data/genes_pleiotropy/.part-00000-bc3ff23c-243e-4360-a61a-61cb7c42af54-c000.snappy.parquet.crc
  
Copying gs://genetics-portal-dev-analysis/dc16/output/genes_pleiotropy/.part-00001-bc3ff23c-243e-4360-a61a-61cb7c42af54-c000.snappy.parquet.crc to file:///Users/polina/Gentropy-manuscript/data/genes_pleiotropy/.part-00001-bc3ff23c-243e-4360-a61a-61cb7c42af54-c000.snappy.parquet.crc
Copying gs://genetics-portal-dev-analysis/dc16/output/genes_pleiotropy/.part-00002-bc3ff23c-243e-4360-a61a-61cb7c42af54-c000.snappy.parquet.crc 

CompletedProcess(args=['gcloud', 'storage', '--billing-project=open-targets-genetics-dev', 'rsync', '-r', 'gs://genetics-portal-dev-analysis/dc16/output/genes_pleiotropy', '/Users/polina/Gentropy-manuscript/data/genes_pleiotropy'], returncode=0)

In [ ]:
def read_genes_pleiotropy_parquet(spark_session, path=None):
    """Read genes_pleiotropy parquet using PySpark and return the DataFrame. Uses local data folder by default."""
    if path is None:
        path = filepath
    return spark_session.read.parquet(path)


# Read from local data folder and show headers (column names)
genes_pleiotropy_df = read_genes_pleiotropy_parquet(session.spark)
print("Headers (columns):", genes_pleiotropy_df.columns)
genes_pleiotropy_df.printSchema()

Headers (columns): ['geneId', 'uniqueVariants', 'uniqueDiseases', 'uniqueTherapeuticAreas', 'maxEQTLColoc', 'maxPQTLColoc', 'maxVEP', 'maxDistanceTSS', 'minEffectiveSampleSize', 'maxEffectiveSampleSize', 'earliestPublicationDate', 'cancerOrBenignTumor', 'infectiousDisease', 'pregnancyOrPerinatalDisease', 'disorderOfVisualSystem', 'cardiovascularDisease', 'pancreasDisease', 'gastrointestinalDisease', 'reproductiveSystemOrBreastDisease', 'integumentarySystemDisease', 'endocrineSystemDisease', 'respiratoryOrThoracicDisease', 'urinarySystemDisease', 'musculoskeletalOrConnectiveTissueDisease', 'disorderOfEar', 'immuneSystemDisease', 'hematologicDisease', 'nervousSystemDisease', 'psychiatricDisorder', 'nutritionalOrMetabolicDisease', 'geneticFamilialOrCongenitalDisease', 'injuryPoisoningOrOtherComplication', 'signOrSymptom', 'other', 'totalStudies', 'approvedSymbol', 'lofConstraint', 'misConstraint', 'synConstraint', 'pathwayCount', 'geneLength', 'tissueSpecificity', 'tissueDistribution', 'n

In [12]:
import pyspark.sql.functions as F

# Z-score: (x - mean) / std over uniqueTherapeuticAreas
stats = genes_pleiotropy_df.agg(
    F.mean("uniqueTherapeuticAreas").alias("mean"),
    F.stddev("uniqueTherapeuticAreas").alias("std"),
).collect()[0]
mean_ta, std_ta = stats["mean"], stats["std"]
if std_ta is None or std_ta == 0:
    std_ta = 1.0

ranked = (
    genes_pleiotropy_df.withColumn(
        "globalScore",
        (F.col("uniqueTherapeuticAreas") - mean_ta) / std_ta,
    )
    .select(
        "globalScore",
        F.col("approvedSymbol").alias("symbol"),
    )
    .orderBy(F.desc("globalScore"))
)

out_path = Path("../../../data/for_gsea/geneset_ta_pleiotropy_zscore.tsv").resolve()
out_path.parent.mkdir(parents=True, exist_ok=True)
ranked.toPandas().to_csv(out_path, sep="\t", index=False)
print(f"Written: {out_path}")

Written: /Users/polina/Gentropy-manuscript/data/for_gsea/geneset_ta_pleiotropy_zscore.tsv


## Background lists

All genes from Reactome and KEGG are used as background for GSEA. They are added to ranked list with Score = 0.

In [18]:
import pandas as pd

gmt_path = Path("../../../data/gene_sets/gene_sets.gmt").resolve()

# Extract all unique genes from the GMT (term\tdesc\tgene1\tgene2\t...)
all_library_genes = set()
with gmt_path.open("r") as f:
    for line in f:
        parts = line.strip().split("\t")
        if len(parts) > 2:
            all_library_genes.update(parts[2:])

ranked_pd = ranked.toPandas()
symbols_in_ranked = set(ranked_pd["symbol"])
# Add library genes that are not already in ranked (with globalScore = 0)
missing_in_ranked = all_library_genes - symbols_in_ranked
background_rows = pd.DataFrame({"globalScore": 0.0, "symbol": list(missing_in_ranked)})
ranked = pd.concat([ranked_pd, background_rows], ignore_index=True).sort_values("globalScore", ascending=False)

out_path = Path("../../../data/for_gsea/geneset_ta_pleiotropy_background.tsv").resolve()
ranked.to_csv(out_path, sep="\t", index=False)
print(f"Added {len(missing_in_ranked)} background genes; total rows: {len(ranked)}. Written: {out_path}")

Added 7477 background genes; total rows: 15762. Written: /Users/polina/Gentropy-manuscript/data/for_gsea/geneset_ta_pleiotropy_background.tsv


# GSEA

GSEA is performed using blitzgsea with pathway library gene sets (Reactome and KEGG) as background.

In [13]:
import os
from pathlib import Path
import pandas as pd
import blitzgsea as blitz

In [14]:
def load_custom_gmt(path):
    """
    Parse a GMT file into a dict: {term_name: [gene1, gene2, ...], …}
    """
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"GMT file not found: {path}")

    with path.open("r") as f:
        return {
            parts[0]: parts[2:]  # skip description at index 1
            for line in f
            if (parts := line.strip().split("	")) and len(parts) > 2
        }


def run_gsea_pandas(
    input_tsv,
    gmt_file,
    output_tsv=None,
    processes=4,
    symbol_col="symbol",
    score_col="globalScore",
):
    """
    Reads TSV, renames columns, runs GSEA with custom pathways, saves as TSV.

    Parameters
    ----------
    input_tsv : str
        Input file with at least the columns specified by symbol_col and score_col.
    gmt_file : str
        Path to custom GMT file.
    processes : int
        Number of processes for GSEA.
    output_tsv : str or None
        Custom output filename. If None, defaults to <input_basename>_gsea.tsv.
    symbol_col : str
        Column name containing gene symbols (default "symbol").
    score_col : str
        Column name containing scores (default "globalScore").
    """
    input_path = Path(input_tsv)
    if not input_path.is_file():
        raise FileNotFoundError(f"Input TSV not found: {input_path}")

    gmt_path = Path(gmt_file)
    if not gmt_path.is_file():
        raise FileNotFoundError(f"GMT file not found: {gmt_path}")

    library_sets = load_custom_gmt(gmt_path)
    if not library_sets:
        raise ValueError(f"No pathways found in {gmt_path}")

    df = pd.read_csv(input_path, sep="	", header=0, index_col=None)

    missing_cols = {symbol_col, score_col} - set(df.columns)
    if missing_cols:
        raise ValueError(f"Missing required columns in input TSV: {sorted(missing_cols)}")

    gsea_df = pd.DataFrame()
    gsea_df[1] = df[symbol_col]
    gsea_df[0] = pd.to_numeric(df[score_col], errors="coerce")
    gsea_df = gsea_df.dropna(subset=[0])

    print(f"GSEA input shape: {gsea_df.shape}")
    print(f"GSEA input columns: {gsea_df.columns.tolist()}")
    print("GSEA input sample:")
    print(gsea_df.head())

    res_df = blitz.gsea(gsea_df, library_sets, processes=processes).reset_index(names="Term")

    res_df["propagated_edge"] = res_df["Term"].apply(
        lambda t: ",".join(library_sets.get(t, [])) if library_sets.get(t) else ""
    )

    term_series = res_df["Term"]
    res_df["Source"] = term_series.str.extract(r"\{([^}]+)\}", expand=False).fillna("")
    res_df["Term"] = term_series.str.replace(r"\{[^}]+\}", "", regex=True)
    res_df["Term"] = res_df["Term"].str.replace(r"\s*\[[^\]]+\]", "", regex=True).str.strip()

    if "leading_edge" in res_df.columns:
        res_df["leading_edge"] = res_df["leading_edge"].apply(
            lambda x: ",".join(map(str, x)) if isinstance(x, (list, tuple)) else str(x)
        )

    first_cols = ["Term", "Source"]
    res_df = res_df[first_cols + [c for c in res_df.columns if c not in first_cols]]

    output_path = Path(output_tsv) if output_tsv is not None else input_path.with_name(f"{input_path.stem}_gsea.tsv")

    res_df.to_csv(output_path, sep="	", index=False)
    print(f"GSEA results saved to {output_path}")
    return res_df

In [20]:
scores = "..//..//..//data/for_gsea/geneset_ta_pleiotropy_background.tsv"
library = "..//..//..//data/gene_sets/gene_sets.gmt"
output_name = "..//..//..//data/gsea_results/geneset_ta_pleiotropy_gsea_with_background.tsv"

run_gsea_pandas(scores, library, processes=4, output_tsv=output_name)

GSEA input shape: (15762, 2)
GSEA input columns: [1, 0]
GSEA input sample:
        1         0
0     FTO  8.899292
1     ABO  8.392908
2  CDKN2B  8.392908
3   SMAD3  6.873753
4    APOE  6.873753
GSEA results saved to ../../../data/gsea_results/geneset_ta_pleiotropy_gsea_with_background.tsv


,Term,Source,es,nes,pval,sidak,fdr,geneset_size,leading_edge,propagated_edge
0,Signal Transduction,Reactome_Pathways_2024,0.486533,7.099741,1.249913e-12,3.071037e-09,3.071037e-09,2613,"CDKN2B,SMAD3,APOE,SH2B3,TCF7L2,VEGFA,ESR1,PPAR...","FNBP1,BAD,ANKFY1,FRS3,FRS2,PRKAB1,PRKAB2,SCD,S..."
1,Mitochondrial Translation,Reactome_Pathways_2024,-0.544946,-6.633299,3.282650e-11,8.065472e-08,4.032736e-08,97,"MRPL33,MRPL3,MTRF1L,MRPS35,MRPL39,TSFM,MRPL9,D...","TUFM,PTCD3,MRPS18C,MRPS18B,MRPS18A,DAP3,ERAL1,..."
2,CD22 Mediated BCR Regulation,Reactome_Pathways_2024,-0.850370,-6.263178,3.772086e-10,9.268011e-07,3.089338e-07,70,"CD79B,LYN","LYN,IGHV3-53,IGHV3-11,IGKV2D-40,IGHV3-13,IGKV5..."
3,Negative Epigenetic Regulation of rRNA Expression,Reactome_Pathways_2024,-0.671978,-5.868844,4.388452e-09,1.078237e-05,2.695607e-06,82,"SUDS3,H2AJ,TAF1B,POLR2H,POLR1B,SAP30BP,SIRT1,G...","H2AZ2,5S RRNA,H2BC9,H2BC4,DNMT3B,H2BC5,H2BC3,H..."
4,Mitochondrial Translation Elongation,Reactome_Pathways_2024,-0.531450,-5.404089,6.513877e-08,1.600331e-04,2.756323e-05,91,"MRPL33,MRPL3,MRPS35,MRPL39,TSFM,MRPL9,DAP3,MRP...","TUFM,TSFM,PTCD3,MRPL28,MRPS18C,MRPS18B,MRPS18A..."
...,...,...,...,...,...,...,...,...,...,...
2452,Sensory Perception,Reactome_Pathways_2024,0.322568,-0.000000,1.000000e+00,1.000000e+00,1.000000e+00,640,"MYH9,KCNJ2,APOE,ATP2B1,ADCY3,APOB,SPTBN1,OR9Q1...","GRXCR1,GRXCR2,OR11G2,OR2L8,CACNA2D2,OR1J4,OR2L..."
2453,Sensing of DNA Double Strand Breaks,Reactome_Pathways_2024,0.500267,-0.000000,1.000000e+00,1.000000e+00,1.000000e+00,6,"RAD50,KPNA2","KAT5,NBN,ATM,MRE11,RAD50,KPNA2"
2454,EPH-ephrin Mediated Repulsion of Cells,Reactome_Pathways_2024,0.305853,-0.000000,1.000000e+00,1.000000e+00,1.000000e+00,51,"EFNA1,FYN,EPHA3,EFNA5,EPHA5,MMP9","EPHA6,EPHA5,LYN,PSENEN,EPHA8,EPHA7,MMP2,MMP9,D..."
2455,Signaling by EGFRvIII in Cancer,Reactome_Pathways_2024,0.409938,-0.000000,1.000000e+00,1.000000e+00,1.000000e+00,15,"PIK3R1,PLCG1","EGF,HSP90AA1,SHC1,GAB1,CBL,EGFR,NRAS,PIK3R1,CD..."
